NSLI 2D and 3D Volterra system with LSI and QSI kernels     

---

Kishore Kumar Tarafdar, Date: 30-06-2025


In [1]:
pwd

'/data1/kishoretarafdar/src.port/VolterraMRAsystems.v0/VolterraSys/nonlinear_kernels.toupdate'

In [1]:
!python --version

Python 3.12.7


        Disable GPU: Force tensorflow to select CPU

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="-1"    
import tensorflow as tf

2025-06-30 05:58:21.138530: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751243301.159868 2490327 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751243301.166485 2490327 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-30 05:58:21.188877: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


        Select a GPU with a memory limit

In [1]:
import tensorflow as tf
print(f"TensorFlow version {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
len(gpus)

2025-06-30 01:19:44.388700: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751226584.410269 2199045 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751226584.416927 2199045 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-30 01:19:44.440919: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version 2.18.0
Num GPUs Available:  3


3

Select one GPU

        Restrict code to use a particular GPU...

In [2]:
# # include ../dirx 
mylibpath = [
    '/home/kishoretarafdar/bin',
    '/data1/kishoretarafdar/src.port/NSLI.v00/utils.Volterra'
    #'/home/k/PLAYGROUND10GB/SKULSTRIPpaper__'
    ]
import sys
[sys.path.insert(0,_) for _ in mylibpath]
del mylibpath

from tf_select_a_gpu import select_a_gpu

In [5]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [3]:
# select_gpu = gpus[gpu_id]
memory_limit = 48#GB
select_a_gpu(gpus, gpu_id=2, memory_limit=memory_limit)
# del gpu_id, select_a_gpu, select_gpu

3 Physical GPUs available 
Selected 1 Logical GPU with 48 GB memory limit


I0000 00:00:1751226591.644289 2199045 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 49152 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6


# Separable conv4d with 2d kernels

        Quadratic NLSI for 2D I/O

    
        Strategy tested with nonseparable conv2d 
        Perfect match with one channel input
        !! Does not match when multiple channel input

        !! Not possible to test the strategy with nonseparable high dimensional convolutions
        (apply update when a libray is located online for nonseparable 4d convolutions)

In [ ]:
#%%
# # # include ../dirx 
mylibpath = [
    '/data1/kishoretarafdar/src.port/multiresolution_kernels.v0/VolterraSys/ndconvolutions'
    ]
import sys
[sys.path.insert(0,_) for _ in mylibpath]
del mylibpath

import tensorflow as tf
from SeparableConv4D import SeparableConv4D
from SeparableConv6D import SeparableConv6D
from trace_blocksum_op import blocksum_op, blocksum3d_op
# from ConvNDv0 import ConvND

class NLSINDVolterra(tf.keras.layers.Layer):
    """ Mother class ND input-output NLSI system with 2nd order Volterra approximation       

        VolterraSys: Multidimensional linear and nonlinear Volterra kernels in natural and multiresolution bases.
        Copyright (C) 2025 Kishore Kumar Tarafdar

        This program is free software: you can redistribute it and/or modify
        it under the terms of the GNU General Public License as published by
        the Free Software Foundation, either version 3 of the License, or
        (at your option) any later version.

        This program is distributed in the hope that it will be useful,
        but WITHOUT ANY WARRANTY; without even the implied warranty of
        MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
        GNU General Public License for more details.

        You should have received a copy of the GNU General Public License
        along with this program.  If not, see <https://www.gnu.org/licenses/>.   

    Limitation: Be careful with the # of filters --kkt@29-06-2025"""
    def __init__(self, filters=1, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size    

    def build(self, input_shape):
        self.inchannels = input_shape[-1]
        # self.filters = input_shape[-1]
        self.ndim = len(input_shape) - 2  # exclude batch and channel dims
        spatial_shape = input_shape[1:-1]  # [N1, N2, N3, ..., ND]

        ### FOR second order input: x*x outer product 
        ## Adding combined support for 2D and 3D
        rank = len(spatial_shape)
        if rank == 1:
            # 1D case: [batch, i, j, p]
            self._einsum_str = 'bip,bjp->bijp'         ## case 1D
        elif rank == 2:
            # 2D case: [batch, i, j, p]
            self._einsum_str = 'bijp,bklp->bijklp'     ## case 2D
        elif rank == 3:
            # 3D case: [batch, i, j, k, c]
            self._einsum_str = 'bijkc,blmnc->bijklmnc' ## case 3D
        else:
            raise ValueError(f"Unsupported input rank: {rank}")

        ## h0:= 0th order term
        # Initialize the bias (a0) for the numerator polynomial
        self.a0 = self.add_weight(shape=(self.filters,), initializer='zeros', trainable=True)#, name='h0_bias')
        self.a1 = self.add_weight(shape=(self.filters,), initializer='zeros', trainable=True)#, name='h0_bias')
        self.a2 = self.add_weight(shape=(self.filters,), initializer='zeros', trainable=True)#, name='h0_bias')

        ## shape match support for LSI kernel
        # Experimental pointwise kernel! a nontrainable pointwise convolution with ones to match 
        # input channels to number of output channels (for smooth separable conv with self.kernel)
        # Determine shape for pointwise kernel (1x1x1...x1, in_channels, out_channels)
        pointwise_shape = (1,) * self.ndim + (self.inchannels, self.filters) 
        self.pointwise = self.add_weight(
            name='match_inchannels_with_outchannels_number',
            # shape=(1, 1, input_shape[-1], self.filters),
            shape = pointwise_shape,
            initializer='ones',
            trainable=False)
        print('ps', self.pointwise.shape)
        ## LSI h:= UIR kernel (multichannel approx)  
        # Create a ND UIR kernel that will be applied to both spatial dimensions
        # Shape for separable spatial kernel: (K, K, ..., K, filters, filters)
        kernel_shape = (self.kernel_size,) * self.ndim + (self.filters, self.filters)
        # kernel_shape = (self.kernel_size,) * ndim + (self.filters, self.filters) ## dependency: SeparableConvND
        self.kernel = self.add_weight(
            name='kernel_UIR',
            # shape=(self.kernel_size, self.kernel_size, input_shape[-1], self.filters),
            shape=kernel_shape,
            initializer='glorot_uniform',
            trainable=True
        )
        print('ks', self.kernel.shape)

        ## QSI op
        ## Adding combined support for 2D and 3D
        if self.ndim == 1:
            # self.kernel = tf.einsum('ico,kco->ikco', self.kernel, self.kernel)
            pass
            # self.separable_conv = separable
        elif self.ndim == 2:   
            self.separable_conv = SeparableConv4D(filters=self.filters, kernel=self.kernel)
            self.blocksum_op = blocksum_op
        elif self.ndim == 3: 
            self.separable_conv = SeparableConv6D(filters=self.filters, kernel=self.kernel)
            self.blocksum_op = blocksum3d_op
        else: raise ValueError(f"This layer support upto SeparableConv6D (D=3) right now!!!")
        

    # def call(self,x):
    #     pass
    

    def get_kernel(self):
        """Returns the kernel weights as a numpy array"""
        return self.kernel.numpy()

    def get_config(self):
        config = super().get_config()
        config.update({
            'filters': self.filters,
            'kernel_size': self.kernel_size
        })
        return config



        2D and 3D inherited kernel

In [7]:
from trace_blocksum_op import blocksum_op, blocksum3d_op

class NLSI2D3DVolterra(NLSINDVolterra):
    """ NSLI 2D and 3D Volterra system with LSI and QSI kernels
    
        VolterraSys: Multidimensional linear and nonlinear Volterra kernels in natural and multiresolution bases.
        Copyright (C) 2025 Kishore Kumar Tarafdar

        This program is free software: you can redistribute it and/or modify
        it under the terms of the GNU General Public License as published by
        the Free Software Foundation, either version 3 of the License, or
        (at your option) any later version.

        This program is distributed in the hope that it will be useful,
        but WITHOUT ANY WARRANTY; without even the implied warranty of
        MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
        GNU General Public License for more details.

        You should have received a copy of the GNU General Public License
        along with this program.  If not, see <https://www.gnu.org/licenses/>.

    Limitation: Be careful with the # of filters --kkt@29-06-2025"""
    def __init__(self, filters=1, kernel_size=3, **kwargs):
        super().__init__(filters=filters, kernel_size=kernel_size, **kwargs)
        # self.separable_conv4d = SeparableConv4D

    def call(self, x):
        input_size = x.shape[1]

        ## outer product of x
        # x2 = tf.einsum('bijp,bklp->bijklp',x,x)
        x2 = tf.einsum(self._einsum_str, x, x)        
        
        # first order coeffs.
        x = tf.nn.convolution(x, self.pointwise, padding='SAME')    ## match inchannels with out channels
        h1_out = tf.nn.convolution(x, self.kernel, padding='SAME')
        
        # 2nd order coeffs
        h2_out = self.separable_conv(x2)
        # print('h1 h2 out', h1_out.shape, h2_out.shape)

        ## collect quadratic coeffs (new upgrade)
        h2_out = self.blocksum_op(h2_out)
        # print('h2_out shape update', h2_out.shape)
              
        # system output
        y = self.a0 + self.a1*h1_out + self.a2*h2_out # + h3_out #+ h4_out + h5_out
        return y

    
    # # Function: 4D separable convolution using a single 2D kernel
    # def __separable_conv4d(self, x):
    #     # x: shape [B, N1, N2, N3, N4, C]
    #     # B, N1, N2, N3, N4, C = x.shape
    #     # Get static shape for dimensions that shouldn't change
    #     input_shape = x.shape.as_list()
    #     N1, N2, N3, N4 = input_shape[1], input_shape[2], input_shape[3], input_shape[4]
        
    #     # Get dynamic batch size
    #     B = tf.shape(x)[0]

    #     ## Step 1: Convolve over (N1, N2)
    #     x1 = tf.reshape(x, [-1, N1, N2, self.inchannels])  # shape: (B*N3*N4, N1, N2, C)
    #     y1 = tf.nn.convolution(x1, self.kernel, padding='SAME')
    #     y1 = tf.reshape(y1, [B, N1, N2, N3, N4, self.filters])   # (B, N1, N2, N3, N4, C)
    #     y1 = tf.transpose(y1, perm=[0,3,4,1,2,5])
    
        
    #     # print('+y1 ', y1.shape)


    #     ## Step 2: Convolve over (N3, N4)
    #     x2 = tf.reshape(y1, [-1, N3, N4, self.filters])  # shape: (B*N1*N2, N3, N4, C)
    #     y2 = tf.nn.convolution(x2, self.kernel, padding='SAME')
    #     y2 = tf.reshape(y2, [B, N1, N2, N3, N4, self.filters])    # final shape
    #     y2 = tf.transpose(y2, perm=[0,3,4,1,2,5])

    #     return y2



if __name__ =='__main__':

    # Define input shape and build the model for summary
    input_shape = (16, 16, 2)  # Replace N with the actual size of x
    inputs = tf.keras.Input(shape=input_shape)

    # Create an instance of the custom layer
    H = NLSI2D3DVolterra(filters=7)
    #conv1d_filters=32, conv1d_kernel_size=3, 
    #  conv2d_filters=32, conv2d_kernel_size=3)

    # Apply the custom layer to the inputs
    outputs = H(inputs)

    # Build the model
    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Print the model summary
    model.summary()

ps (1, 1, 2, 7)
ks (3, 3, 7, 7)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 16, 16, 2)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nlsi2d3d_volterra_3             │ (None, 16, 16, 7)      │           490 │
│ (NLSI2D3DVolterra)              │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 490 (1.91 KB)

 Trainable params: 462 (1.80 KB)

 Non-trainable params: 28 (112.00 B)

In [8]:
if __name__ =='__main__':

    # Define input shape and build the model for summary
    input_shape = (16, 16, 16, 2)  # Replace N with the actual size of x
    inputs = tf.keras.Input(shape=input_shape)

    # Create an instance of the custom layer
    H = NLSI2D3DVolterra(filters=13)
    #conv1d_filters=32, conv1d_kernel_size=3, 
    #  conv2d_filters=32, conv2d_kernel_size=3)

    # Apply the custom layer to the inputs
    outputs = H(inputs)

    # Build the model
    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Print the model summary
    model.summary()

ps (1, 1, 1, 2, 13)
ks (3, 3, 3, 13, 13)
kernel (3, 3, 3, 13, 13)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 16, 16, 16, 2)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nlsi2d3d_volterra_4             │ (None, 16, 16, 16, 13) │         4,654 │
│ (NLSI2D3DVolterra)              │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,654 (18.18 KB)

 Trainable params: 4,602 (17.98 KB)

 Non-trainable params: 52 (208.00 B)

        OK till above
        
        2D and 3D done

---

In [ ]:
class TraceConv1Dio(tf.keras.layers.Layer):
    """Trace Convolution: Covolution with strides along the trace of the tensor

       Wiener System / LN cascade with Volterra series
       Input: Sequence x[n]
       Output: Shift invariant quadratic monomial q_2, i.e., m=2

       --@KKT, 7-Feb-2025
    
    """
    
    def __init__(self, kernel_size=4, **kwargs):
        super().__init__(**kwargs)
        self.L = kernel_size
        
    
    def build(self, input_shape):
        # input_shape: (batch_size, N, N, channels)
        self.N = input_shape[1]
        channels = input_shape[-1]
        
        
        # # Initialize learnable kernel
        # self.h2 = self.add_weight(
        #     name='h2_kernel',
        #     shape=(self.N, self.N, channels),
        #     initializer='glorot_uniform',
        #     trainable=True
        # )
        
        # Precompute circular indices
        ## FULL LENGTH FILTER: kernel_size = input_size
       
        # self.row_indices = tf.math.mod(n[:, tf.newaxis] - n, self.N)
        n = tf.range(self.N)
        self.row_indices = tf.math.mod(n[:, tf.newaxis] - tf.range(self.L), self.L)
        
        ## SMALL LENGTH FILTER: kernel_size < input_size
        
        super().build(input_shape)
    
   
   
   
   
    def call(self, x2, h2):
        """
        Args:
            inputs: Tensor of shape (batch_size, N, N, channels)
        Returns:
            Tensor of shape (batch_size, N)
        """
        self.h2 = h2
        print('xh+',x2.shape, self.h2.shape)





        ### PAD kernel with zeros to match the input shape
        paddings = [[0, self.N - self.kernel_size] for _ in range(self.dim)]  # spatial dims
        paddings += [[0, 0], [0, 0]]  # in_channels, out_channels
        print('dim', self.dim)
        print(paddings)
        print(self.kernel.shape)
        # kernel_padded = tf.pad(self.kernel, paddings, mode='CONSTANT', constant_values=0)
        kernel_padded = tf.pad(h2, paddings, mode='CONSTANT', constant_values=0)
        


        
        
        
        
        
        
        
        batch_size = tf.shape(x2)[0]
        # print('r+',self.row_indices.shape)
        
        # Expand indices for batch dimension
        batch_indices = tf.expand_dims(self.row_indices, 0)  # (1, N, N)
        # print('bi+',batch_indices.shape)
        batch_indices = tf.tile(batch_indices, [batch_size, 1, 1])  # (batch, N, N)
        # print('bi+',batch_indices.shape)

        # Gather shifted rows and columns
        rows_shifted = tf.gather(x2, self.row_indices, axis=1)  # (batch, N, N, N, ch)
        columns_shifted = tf.gather(
            rows_shifted, 
            batch_indices,  # Use batch-aware indices
            axis=3, 
            batch_dims=2    # Match batch dimension
        )  # (batch, N, N, N, ch)
        print('rc+',rows_shifted.shape, columns_shifted.shape, self.h2.shape)
 
        
        
        
        
        
        
        
        result = tf.einsum('bnijc,ijc->bnc', columns_shifted, self.h2)
        return result
    
    
    
    
    
    
    
    
    
    
    def get_config(self):
        base_config = super().get_config()
        return base_config




class NLSI1DVolterra(NLSINDVolterra):
    """ NSLI 2D and 3D Volterra system with LSI and QSI kernels
    
        VolterraSys: Multidimensional linear and nonlinear Volterra kernels in natural and multiresolution bases.
        Copyright (C) 2025 Kishore Kumar Tarafdar

        This program is free software: you can redistribute it and/or modify
        it under the terms of the GNU General Public License as published by
        the Free Software Foundation, either version 3 of the License, or
        (at your option) any later version.

        This program is distributed in the hope that it will be useful,
        but WITHOUT ANY WARRANTY; without even the implied warranty of
        MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
        GNU General Public License for more details.

        You should have received a copy of the GNU General Public License
        along with this program.  If not, see <https://www.gnu.org/licenses/>.

    Limitation: Be careful with the # of filters --kkt@29-06-2025"""
    def __init__(self, filters=1, kernel_size=3, **kwargs):
        super().__init__(filters=filters, kernel_size=kernel_size, **kwargs)
        # self.separable_conv4d = SeparableConv4D

    def call(self, x):
        input_size = x.shape[1]
        
        ## outer product of x
        # x2 = tf.einsum('bijp,bklp->bijklp',x,x)
        x2 = tf.einsum(self._einsum_str, x, x)        
        
        
        ## LSI
        x = tf.nn.convolution(x, self.pointwise, padding='SAME')    ## match inchannels with out channels
        h1_out = tf.nn.convolution(x, self.kernel, padding='SAME')
        
        ## QSI
        kernel = tf.einsum('ico,kco->ikco', self.kernel, self.kernel)
        h2_out = tf.nn.convolution(x2, kernel, padding='SAME')
        






        y = self.a0 + self.a1*h1_out + self.a2*h2_out # + h3_out #+ h4_out + h5_out
        return y

if __name__ =='__main__':

    # Define input shape and build the model for summary
    input_shape = (16, 2)  # Replace N with the actual size of x
    inputs = tf.keras.Input(shape=input_shape)

    # Create an instance of the custom layer
    H = NLSI1DVolterra(filters=7)
    #conv1d_filters=32, conv1d_kernel_size=3, 
    #  conv2d_filters=32, conv2d_kernel_size=3)

    # Apply the custom layer to the inputs
    outputs = H(inputs)

    # Build the model
    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Print the model summary
    model.summary()

ks (3, 3, 3, 2, 8)
  x  (None, 16, 16, 16, 2)
+x2 (None, 16, 16, 16, 16, 16, 16, 2)
kernel (3, 3, 3, 2, 8)
h1 h2 out (None, 16, 16, 16, 8) (None, 16, 16, 16, 16, 16, 16, 8)
h2_out shape update (None, 16, 16, 16, 8)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 16, 16, 16, 2)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nlsi2d3d_volterra_1             │ (None, 16, 16, 16, 8)  │           472 │
│ (NLSI2D3DVolterra)              │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 472 (1.84 KB)

 Trainable params: 456 (1.78 KB)

 Non-trainable params: 16 (64.00 B)

In [20]:
class TraceConv1Dio(tf.keras.layers.Layer):
    """Trace Convolution: Covolution with strides along the trace of the tensor

       Wiener System / LN cascade with Volterra series
       Input: Sequence x[n]
       Output: Shift invariant quadratic monomial q_2, i.e., m=2

       --@KKT, 7-Feb-2025
    
    """
    
    def __init__(self, kernel, **kwargs):
        super().__init__(**kwargs)
        self.h2 = kernel
        self.L = tf.shape(kernel)[0]
    
    def build(self, input_shape):
        # input_shape: (batch_size, N, N, channels)
        self.N = input_shape[1]
        channels = input_shape[-1]
        
        # # Initialize learnable kernel
        # self.h2 = self.add_weight(
        #     name='h2_kernel',
        #     shape=(self.N, self.N, channels),
        #     initializer='glorot_uniform',
        #     trainable=True
        # )
        
        # Precompute circular indices
        ## FULL LENGTH FILTER: kernel_size = input_size
        n = tf.range(self.N)
        # self.row_indices = tf.math.mod(n[:, tf.newaxis] - n, self.N)
        
        ## SMALL LENGTH FILTER: kernel_size < input_size
        self.row_indices = tf.math.mod(n[:, tf.newaxis] - tf.range(self.L), self.L)
        super().build(input_shape)
    
    def call(self, x2):
        """
        Args:
            inputs: Tensor of shape (batch_size, N, N, channels)
        Returns:
            Tensor of shape (batch_size, N)
        """
        # self.h2 = h2
        print('xh+',x2.shape, self.h2.shape)
        batch_size = tf.shape(x2)[0]
        # print('r+',self.row_indices.shape)
        # Expand indices for batch dimension
        batch_indices = tf.expand_dims(self.row_indices, 0)  # (1, N, N)
        # print('bi+',batch_indices.shape)
        batch_indices = tf.tile(batch_indices, [batch_size, 1, 1])  # (batch, N, N)
        # print('bi+',batch_indices.shape)

        # Gather shifted rows and columns
        rows_shifted = tf.gather(x2, self.row_indices, axis=1)  # (batch, N, N, N, ch)
        columns_shifted = tf.gather(
            rows_shifted, 
            batch_indices,  # Use batch-aware indices
            axis=3, 
            batch_dims=2    # Match batch dimension
        )  # (batch, N, N, N, ch)
        print('rc+',rows_shifted.shape, columns_shifted.shape, self.h2.shape)
 
        result = tf.einsum('bnijc,ijc->bnc', columns_shifted, self.h2)
        return result
    
    def get_config(self):
        base_config = super().get_config()
        return base_config

import numpy as np
x = np.array([1, 2, 3,4,5])
h2 = np.array([[1, 2], 
              [1, 3]])
# h2 = np.array([[1, 1, -1], 
#               [1, 2, -1], 
#               [1, 3, -1]])

x2 = np.einsum('i,j->ij', x, x)
y = TraceConv1Dio(kernel=tf.expand_dims(h2, axis=-1))(tf.expand_dims(tf.expand_dims(x2,axis=0),axis=-1))
y.shape

# if __name__ =='__main__':

#     # Define input shape and build the model for summary
#     input_shape = (16, 1)  # Replace N with the actual size of x
#     inputs = tf.keras.Input(shape=input_shape)

#     # Create an instance of the custom layer
#     h2 = np.array([[1, 2], 
#               [1, 3]]).astype(np.float32)
#     H = TraceConv1Dio(kernel=tf.expand_dims(h2, axis=-1))
#     # (tf.expand_dims(tf.expand_dims(x2,axis=0),axis=-1))

#     #conv1d_filters=32, conv1d_kernel_size=3, 
#     #  conv2d_filters=32, conv2d_kernel_size=3)

#     # Apply the custom layer to the inputs
#     outputs = H(inputs)

#     # Build the model
#     model = tf.keras.Model(inputs=inputs, outputs=outputs)

#     # Print the model summary
#     model.summary()


xh+ (1, 5, 5, 1) (2, 2, 1)
rc+ (1, 5, 2, 5, 1) (1, 5, 2, 2, 1) (2, 2, 1)


TensorShape([1, 5, 1])

In [11]:
tf.expand_dims(h2, axis=-1).shape

TensorShape([2, 2, 1])

    prereq. check blocksum

In [7]:
import tensorflow as tf

# Replace these with your desired values
batch, n, channels = 12, 32, 1 

shape = (batch, n, n, n, n, channels)
x = tf.random.uniform(shape=shape, minval=0.0, maxval=1.0)
print(x.shape)

(12, 32, 32, 32, 32, 1)


In [8]:
from trace_blocksum_op import blocksum_op
dir(blocksum_op)
# print(trace_blocksum_op.blocksum_op)
# print(trace_blocksum_op.blocksum3d_op)

['__annotations__',
 '__builtins__',
 '__call__',
 '__class__',
 '__closure__',
 '__code__',
 '__defaults__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__globals__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__kwdefaults__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__type_params__',
 '_tf_api_names',
 '_tf_api_names_v1',
 '_tf_fallback_dispatchers',
 '_tf_type_based_dispatcher']

In [9]:
import tensorflow as tf

# Replace these with your desired values
batch, n, channels = 12, 32, 1 

shape = (batch, n, n, n, n, channels)
x = tf.random.uniform(shape=shape, minval=0.0, maxval=1.0)
print(x.shape)

(12, 32, 32, 32, 32, 1)


In [10]:
blocksum_op(x)

<tf.Tensor: shape=(12, 32, 32, 1), dtype=float32, numpy=
array([[[[5.7894433e-01],
         [1.7053549e+00],
         [4.1432953e+00],
         ...,
         [4.4567813e+02],
         [4.7794016e+02],
         [5.0569516e+02]],

        [[2.0962963e+00],
         [7.2039709e+00],
         [1.5541157e+01],
         ...,
         [1.7894829e+03],
         [1.9157898e+03],
         [2.0358929e+03]],

        [[5.2488561e+00],
         [1.8926706e+01],
         [4.0575146e+01],
         ...,
         [4.0323572e+03],
         [4.3094155e+03],
         [4.5814639e+03]],

        ...,

        [[4.6799179e+02],
         [1.8213878e+03],
         [4.0777126e+03],
         ...,
         [4.0499128e+05],
         [4.3255153e+05],
         [4.6095694e+05]],

        [[4.9591074e+02],
         [1.9318293e+03],
         [4.3355288e+03],
         ...,
         [4.3240984e+05],
         [4.6186369e+05],
         [4.9220419e+05]],

        [[5.2765900e+02],
         [2.0542014e+03],
         [4.61529

In [3]:
# import trace_blocksum_op
# import tensorflow as tf
# import importlib.resources as pkg_resources
# # import myop  # this is the installed package

# # Use importlib to find the path to the .so inside the installed package
# with pkg_resources.path(trace_blocksum_op, "blocksum_op_noxla_v001.so") as so_path:
#     myop = tf.load_op_library(str(so_path))

# # Now use your op
# output = myop.blockwise_diagonal_sum(...)
